# Cluster browser

Interactive notebook to browse single-unit rasters and PSTHs. Use the widgets below to pick a cluster, alignment event, time window, and smoothing.

Requirements: `ipywidgets`, `matplotlib`, `numpy`, `pandas`, and the repository package installed (editable).

In [55]:
# Cell 1: imports and env check
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

repo_root = Path('..').resolve()
sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)

# load helper functions from src
from src.session_io import load_session
from src.su_analysis import plot_raster_psth
from src.align import get_event_times

print('Imports OK')

Repo root: E:\python_analysis\git_repos\vis_detect_analysis_Sep2025
Imports OK


In [56]:
# Cell 2: load session
session_pkl = Path('..') / 'data' / 'BG_046_15082025.pkl'
session = load_session(str(session_pkl))
print('Loaded session:', session.subject, session.session_name)

# gather cluster ids
cluster_ids = [int(c.cluster_id) for c in session.clusters]
print('N clusters:', len(cluster_ids))

Loaded session: BG_046 15082025
N clusters: 622


In [57]:
# Cell 3: widgets
# checkbox to toggle viewing only 'good' clusters
require_good = widgets.Checkbox(value=True, description='Good-only')
cluster_select = widgets.Dropdown(options=cluster_ids, description='Cluster')
event_select = widgets.Dropdown(options=['Change_ON','Baseline_ON'], description='Event')
window_slider = widgets.FloatRangeSlider(value=[-0.5,1.0], min=-2.0, max=2.0, step=0.01, description='Window')
bin_size = widgets.FloatSlider(value=0.01, min=0.001, max=0.1, step=0.001, description='Bin size')
smooth = widgets.IntSlider(value=2, min=0, max=20, step=1, description='Smooth (bins)')
update_btn = widgets.Button(description='Update plot')

controls = widgets.VBox([require_good, cluster_select, event_select, window_slider, bin_size, smooth, update_btn])
display(controls)

# function to update the cluster dropdown based on the checkbox
def _update_cluster_options(change=None):
    good = set(getattr(session, 'good_cluster_ids', []))
    if require_good.value:
        opts = sorted(int(c.cluster_id) for c in session.clusters if int(c.cluster_id) in good)
    else:
        opts = sorted(int(c.cluster_id) for c in session.clusters)
    # preserve current selection if possible
    cur = cluster_select.value
    cluster_select.options = opts
    if cur in opts:
        cluster_select.value = cur

# observe the checkbox and initialize
require_good.observe(_update_cluster_options, names='value')
_update_cluster_options()

In [58]:
# Cell 4: improved plotting callback
out = widgets.Output()
display(out)

# additional controls: trial sort, trial limit
sort_select = widgets.Dropdown(options=['None','RT','outcome'], value='None', description='Sort by')
limit_trials = widgets.IntSlider(value=200, min=10, max=1000, step=10, description='Max trials')
# option: include trials missing the selected event (e.g., FA trials for Change_ON)
include_missing_event = widgets.Checkbox(value=False, description='Include trials missing event')
controls.children = list(controls.children) + [sort_select, limit_trials, include_missing_event]

def smooth_psth(arr, kernel_sigma_bins=2):
    if kernel_sigma_bins <= 0:
        return arr
    # gaussian kernel
    from math import ceil
    sigma = kernel_sigma_bins
    radius = int(ceil(4 * sigma))
    x = np.arange(-radius, radius+1)
    kern = np.exp(-0.5 * (x / float(sigma))**2)
    kern = kern / kern.sum()
    return np.convolve(arr, kern, mode='same')

# canonical mapping for possibly messy outcome labels
_outcome_normalizer = {
    'hit': 'Hit',
    'h': 'Hit',
    'fa': 'FA',
    'false alarm': 'FA',
    'false_alarm': 'FA',
    'falsealarm': 'FA',
    'false-alarm': 'FA',
    'miss': 'Miss',
    'm': 'Miss',
    'abort': 'Abort',
    'aborted': 'Abort',
    'none': None,
    'nan': None,
    '': None,
}

def _canonical_outcome(raw):
    if raw is None:
        return None
    s = str(raw).strip()
    if s == '':
        return None
    key = s.lower()
    return _outcome_normalizer.get(key, s)

from matplotlib.patches import Patch

def on_update(_=None):
    with out:
        clear_output(wait=True)
        cid = int(cluster_select.value)
        event = event_select.value
        w = tuple(window_slider.value)
        bs = float(bin_size.value)
        s = int(smooth.value)
        sort_by = sort_select.value
        max_trials = int(limit_trials.value)
        include_missing = bool(include_missing_event.value)
        print(f'Plotting cluster {cid}, event {event}, window={w}, bin={bs}, smooth={s}, sort={sort_by}, max_trials={max_trials}, include_missing={include_missing}')
        # find cluster spikes
        cluster = next((c for c in session.clusters if int(c.cluster_id)==cid), None)
        if cluster is None:
            print('Cluster not found')
            return
        st = np.asarray(cluster.spike_times).flatten()
        event_times = get_event_times(session, event)
        n_trials = min(len(event_times), len(session.trials))
        # gather per-trial spike times within window; optionally skip trials missing the event
        trials_spikes = []
        trial_outcomes = []
        trial_rt = []
        included_idxs = []
        for ti in range(n_trials):
            et = event_times[ti]
            raw_out = getattr(session.trials[ti], 'trialoutcome', None)
            outcome = _canonical_outcome(raw_out)
            # if aligning to Change_ON and trial is a False Alarm (FA) or Abort, exclude unless include_missing is True
            if event == 'Change_ON' and outcome in ('FA','Abort') and not include_missing:
                # skip FA/Abort when aligning to Change_ON by default
                continue
            if et is None or (isinstance(et, float) and np.isnan(et)):
                if not include_missing:
                    # count as excluded
                    continue
                # include as empty-aligned trial
                trials_spikes.append(np.array([]))
                trial_outcomes.append(outcome)
                trial_rt.append(np.nan)
                included_idxs.append(ti)
                continue
            rel = st - float(et)
            mask = (rel >= w[0]) & (rel <= w[1])
            trials_spikes.append(np.asarray(rel[mask]))
            trial_outcomes.append(outcome)
            # attempt to read reaction time if present on trial (common names: 'rt','reaction_time','rt_true')
            rt = getattr(session.trials[ti], 'rt', None) or getattr(session.trials[ti], 'reaction_time', None) or getattr(session.trials[ti], 'rt_true', None) or np.nan
            trial_rt.append(rt if rt is not None else np.nan)
            included_idxs.append(ti)
        # summary counts
        from collections import Counter
        outcome_counts = Counter([o if o is not None else 'None' for o in trial_outcomes])
        excluded_count = (n_trials - len(included_idxs))
        print('Outcome counts (included):', dict(outcome_counts), 'Excluded trials:', int(excluded_count))
        # build index ordering (over included trials)
        indices = np.arange(len(included_idxs))
        if sort_by == 'RT':
            # sort by reaction time (nan last)
            rt_arr = np.array([np.nan if x is None else x for x in trial_rt], dtype=float)
            order = np.argsort(np.nan_to_num(rt_arr, nan=np.inf))
            indices = order
        elif sort_by == 'outcome':
            # group by outcome: Hit, FA, Miss, Abort (specified canonical order)
            outcome_order = {'Hit':0, 'FA':1, 'Miss':2, 'Abort':3, None:4}
            keys = [outcome_order.get(o, 4) for o in trial_outcomes]
            indices = np.argsort(keys)
        # apply trial limit
        if max_trials and len(indices) > max_trials:
            indices = indices[:max_trials]
        # create figure with raster and PSTH
        fig = plt.figure(figsize=(10,6))
        gs = fig.add_gridspec(2,2, height_ratios=[3,1])
        ax_r = fig.add_subplot(gs[0,:])
        ax_p = fig.add_subplot(gs[1,:])
        # raster: plot each trial's spikes
        yticks = []
        ylabels = []
        # define canonical color mapping for outcomes
        color_map = {
            'Hit': '#2ca02c',    # green
            'FA': '#d62728',     # red (false alarm)
            'Miss': '#7f7f7f',   # gray
            'Abort': '#c49bd6'   # lavender/purple
        }
        # improved visibility: thicker lines and slight alpha
        vline_kwargs = dict(linewidth=1.2, alpha=1.0)
        for row, idx in enumerate(indices):
            ti = int(idx)
            sp = trials_spikes[ti]
            outcome = trial_outcomes[ti]
            # fallback color for unexpected labels
            color = color_map.get(outcome, '0.6')
            if sp.size > 0:
                ax_r.vlines(sp, row + 0.5, row + 1.5, color=color, **vline_kwargs)
            yticks.append(row + 1)
            # show original trial index for reference
            ylabels.append(str(int(included_idxs[ti])))
        ax_r.set_ylim(0.5, len(indices) + 0.5)
        ax_r.set_yticks([])
        ax_r.set_ylabel('Trials')
        ax_r.axvline(0, color='k', linestyle='--')
        ax_r.set_title(f'Cluster {cid} raster (n={len(indices)})')
        # add small legend with counts
        legend_patches = []
        for k, col in color_map.items():
            cnt = outcome_counts.get(k, 0)
            legend_patches.append(Patch(facecolor=col, label=f"{k} ({cnt})"))
        if outcome_counts:
            ax_r.legend(handles=legend_patches, bbox_to_anchor=(1.02, 1), loc='upper left')
        # PSTH: compute binned firing rates across displayed trials
        bins = np.arange(w[0], w[1] + bs, bs)
        counts = np.zeros((len(indices), len(bins)-1), dtype=float)
        for r, idx in enumerate(indices):
            ti = int(idx)
            sp = trials_spikes[ti]
            if sp.size > 0:
                c, _ = np.histogram(sp, bins=bins)
                counts[r, :] = c.astype(float) / bs
            else:
                counts[r, :] = 0.0
        mean_psth = np.nanmean(counts, axis=0) if counts.size else np.zeros(len(bins)-1)
        smooth_p = smooth_psth(mean_psth, kernel_sigma_bins=s)
        centers = (bins[:-1] + bins[1:]) / 2.0
        # plot PSTH in same color as Hit for emphasis
        ax_p.plot(centers, smooth_p, color=color_map['Hit'])
        ax_p.axvline(0, color='k', linestyle='--')
        ax_p.set_xlabel('Time (s)')
        ax_p.set_ylabel('FR (Hz)')
        plt.tight_layout()
        plt.show()

# connect callback
update_btn.on_click(on_update)
# run once to populate
on_update()

Output()